# Importing libraries 

In [1]:
## Importing libraries 

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [2]:
## Set the path and check the dataset

import os

data_path = "PetImages"

print("Classes:", os.listdir(data_path))
print("Cat images:", len(os.listdir(data_path + "/Cat")))
print("Dog images:", len(os.listdir(data_path + "/Dog")))

## Check for corrupted images safely

from PIL import Image

for class_name in ["Cat", "Dog"]:
    folder = os.path.join(data_path, class_name)
    corrupted = 0

    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)

        try:
            img = Image.open(path)
            img.verify()
        except:
            corrupted += 1

    print(class_name, "corrupted images:", corrupted)



Classes: ['Cat', 'Dog']
Cat images: 12499
Dog images: 12499
Cat corrupted images: 0
Dog corrupted images: 0


D:\conda_envs\pytorch\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [3]:
## Remove corrupted images

from PIL import Image
import os

for class_name in ["Cat", "Dog"]:
    folder = os.path.join(data_path, class_name)

    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)

        try:
            img = Image.open(path)
            img.verify()
        except:
            os.remove(path)

print("Corrupted images removed")

Corrupted images removed


# Loading dataset

In [4]:
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

dataset = torchvision.datasets.ImageFolder(
    root = "PetImages",
    transform=transform
)



print("Classes:", dataset.classes)
print("Total images:", len(dataset))

Classes: ['Cat', 'Dog']
Total images: 24998


# Splitting dataset

In [5]:
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

valloader = DataLoader(val_dataset, batch_size=32, shuffle=False)
testloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Data is splitted")

Data is splitted


# Create the DataLoader 

In [6]:
trainloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

print("DataLoader created!")

DataLoader created!


# Define the CNN

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x


model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model is defined")



Model is defined


# Training and validating the CNN

In [8]:
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in valloader:

            output = model(images)
            _, predicted = torch.max(output, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    validation_accuracy = 100 * correct / total

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")
    print("Validation Accuracy:", round(validation_accuracy, 2), "%")

print("Training and validating data is completed")

epoch = 1/10 & loss = 0.6063117947800994
Validation Accuracy: 75.19 %
epoch = 2/10 & loss = 0.4968695114450077
Validation Accuracy: 78.39 %
epoch = 3/10 & loss = 0.4374309890067486
Validation Accuracy: 83.04 %
epoch = 4/10 & loss = 0.3939912329282602
Validation Accuracy: 84.66 %
epoch = 5/10 & loss = 0.35513393662850873
Validation Accuracy: 87.76 %
epoch = 6/10 & loss = 0.32177059783045286
Validation Accuracy: 89.06 %
epoch = 7/10 & loss = 0.2837943544377909
Validation Accuracy: 90.53 %
epoch = 8/10 & loss = 0.24898029235013
Validation Accuracy: 88.98 %
epoch = 9/10 & loss = 0.2140215816990951
Validation Accuracy: 93.87 %
epoch = 10/10 & loss = 0.18401283655992096
Validation Accuracy: 93.65 %
Training and validating data is completed


## Evaluating 

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in testloader:
        output = model(images)
        _, predicted = torch.max(output, 1)
        y_true.extend(labels.tolist())
        y_pred.extend(predicted.tolist())

print("Accuracy : ", accuracy_score(y_true, y_pred)*100,"%")
print("Precision : ", precision_score(y_true, y_pred, zero_division=0)*100,"%")
print("Recall : ", recall_score(y_true, y_pred, zero_division=0)*100,"%")
print("Confusion Matrix :\n", confusion_matrix(y_true, y_pred))

Accuracy :  93.12183417755266 %
Precision :  96.76503972758229 %
Recall :  89.45435466946485 %
Confusion Matrix :
 [[1788   57]
 [ 201 1705]]
